# 17 — Robustness and sensitivity

Test whether conclusions depend on the event definition, transition-year treatment, product definition or source.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from portugal_refining_resilience.config import get_paths
from portugal_refining_resilience.io import persist_dataframe, write_json

PATHS = get_paths(ROOT)
pd.set_option("display.max_columns", 100)

from portugal_refining_resilience.config import load_analysis_config
from portugal_refining_resilience.dgeg import ReconciliationThresholds, compare_trade_sources
from portugal_refining_resilience.metrics import event_window_summary


In [ ]:
panel = pd.read_csv(PATHS.processed / "fuel_annual_analytical_panel.csv")
rows = []
for event_year in [2021, 2022]:
    for pre_years, post_years in [(3, 2), (5, 3), (8, 3)]:
        for metric in ["exports_kt", "net_import_to_demand_ratio", "refinery_output_to_demand_ratio"]:
            if metric not in panel.columns:
                continue
            summary = event_window_summary(panel, value_column=metric, event_year=event_year, pre_years=pre_years, post_years=post_years)
            summary["pre_years"] = pre_years
            summary["post_years"] = post_years
            rows.append(summary)
robust = pd.concat(rows, ignore_index=True)
persist_dataframe(robust, PATHS.metrics / "event_window_sensitivity.csv")
display(robust.head(20))


In [ ]:
# Source reconciliation hook: compare JODI annual trade with a DGEG-extracted canonical file when available.
dgeg_path = PATHS.interim / "dgeg_trade_annual_canonical.csv"
if dgeg_path.exists():
    dgeg = pd.read_csv(dgeg_path)
    jodi = pd.read_csv(PATHS.processed / "fuel_trade_annual.csv")
    # Tolerances live in config/analysis.yml so that changing them is a reviewable
    # decision rather than an edit buried in a notebook. A row is flagged only when
    # it breaches both limits; see data/contracts.md for why.
    recon_config = load_analysis_config(ROOT)["source_reconciliation"]
    comparison = compare_trade_sources(
        jodi,
        dgeg,
        thresholds=ReconciliationThresholds(
            warning_abs_kt=float(recon_config["warning_abs_kt"]),
            warning_pct=float(recon_config["warning_pct"]),
        ),
    )
    persist_dataframe(comparison, PATHS.metrics / "jodi_dgeg_trade_reconciliation.csv", key_columns=["year", "product", "flow"])
    display(comparison.groupby(["product", "flow"])["difference_pct_comparison"].describe())
else:
    print("DGEG canonical trade extraction not yet present; source reconciliation remains incomplete.")


In [ ]:
# Annual source sensitivity. Where the reconciliation flags a trade cell, the annual
# event evidence that depends on it is recomputed on the corroborated national value,
# so no headline annual claim rests on the series identified as the outlier.
from portugal_refining_resilience.breaks import interrupted_time_series
from portugal_refining_resilience.metrics import safe_ratio

recon_path = PATHS.metrics / "jodi_dgeg_trade_reconciliation.csv"
panel_path = PATHS.processed / "fuel_annual_analytical_panel.csv"
rows = []
if recon_path.exists() and panel_path.exists():
    recon = pd.read_csv(recon_path)
    flagged = recon.loc[recon["reconciliation_status"].ne("within_tolerance")]
    annual = pd.read_csv(panel_path)
    for product, group in flagged.groupby("product"):
        base = annual.loc[annual["product"].eq(product)].copy()
        if base.empty:
            continue
        alt = base.copy()
        for row in group.itertuples():
            column = f"{row.flow}_kt"
            if column in alt.columns:
                alt.loc[alt["year"].eq(row.year), column] = float(row.value_kt_dgeg)
        alt["net_imports_kt"] = alt["imports_kt"] - alt["exports_kt"]
        alt["net_import_to_demand_ratio"] = safe_ratio(alt["net_imports_kt"], alt["demand_kt"])
        for outcome in ["exports_kt", "net_import_to_demand_ratio"]:
            for label, frame in (("primary", base), ("corroborated", alt)):
                if frame[outcome].notna().sum() < 10:
                    continue
                model = interrupted_time_series(
                    frame, value_column=outcome, event_year=2022, transition_years=(2021,)
                )
                rows.append({
                    "product": product, "outcome": outcome, "event_year": 2022,
                    "trade_source": label,
                    "level_change": float(model.params["post"]),
                    "std_error": float(model.bse["post"]),
                    "p_value": float(model.pvalues["post"]),
                    "nobs": int(model.nobs),
                })
annual_source_sensitivity = pd.DataFrame(rows)
if not annual_source_sensitivity.empty:
    persist_dataframe(
        annual_source_sensitivity,
        PATHS.metrics / "annual_source_sensitivity.csv",
        key_columns=["product", "outcome", "event_year", "trade_source"],
    )
    display(annual_source_sensitivity)
else:
    print("No flagged reconciliation cells; annual source sensitivity not required.")